# RoomSync — ML Roommate Matching System
## Model Design, Dataset, Training & Evaluation

---

## 1. Problem Statement

Given a student's structured profile (budget, sleep schedule, study habits, cleanliness level, year of study, class schedule), 
recommend the **top-K most compatible roommates** from a pool of registered students.

This is a **recommendation problem** combining:
- **Content-based filtering** (profile similarity)
- **Clustering** (lifestyle grouping)
- **Feedback-based reranking** (future phase)

---

## 2. Dataset

### 2a. Synthetic Dataset (Current)
We generate **500 student profiles** with realistic correlated traits:
- Early birds tend to prefer silent study and score higher on cleanliness
- Night owls tend toward group study with lower cleanliness ratings
- Budget correlates with year of study (seniors have higher budgets)

Generated files:
- `data/student_profiles.csv` — 500 profiles
- `data/compatibility_labels.csv` — 2,000 labeled pairs

### 2b. Real Kaggle Datasets (For Pre-training / Validation)
The following Kaggle datasets can extend our training data:

| Dataset | URL | How Used |
|---------|-----|----------|
| Student Lifestyle Dataset | kaggle.com/datasets/steve1215rogg/student-lifestyle-dataset | Sleep/study patterns |
| Dormitory Preference Survey | (collect via Google Forms) | Labeled pairs |
| MovieLens (collaborative filtering) | grouplens.org/datasets/movielens/ | CF algorithm inspiration |

The MovieLens approach inspires our **collaborative filtering extension**: 
students who liked roommates similar to you → also liked roommate X.

---

## 3. Feature Engineering

| Raw Feature | Encoding | Scale |
|------------|----------|-------|
| sleep_schedule | Label Enc (Early=0, Flex=1, Night=2) | MinMaxScaler |
| study_type | Label Enc (Silent=0, Both=1, Group=2) | MinMaxScaler |
| year | Ordinal (FY=1, SY=2, TY=3, Final=4) | MinMaxScaler |
| schedule | Label Enc (Morning=0..Evening=2, Mixed=3) | MinMaxScaler |
| cleanliness | Integer 1–5 | MinMaxScaler |
| budget | Midpoint (min+max)/2 | MinMaxScaler |
| past_rating | Float 1.0–5.0 | MinMaxScaler |

After scaling, features are **weighted** before cosine similarity:

```python
FEATURE_WEIGHTS = [3.0, 2.5, 1.0, 1.0, 2.0, 4.0, 1.5]
# sleep, study, year, schedule, cleanliness, budget, rating
```

---

## 4. Algorithms

### Algorithm 1 — Cosine Similarity
Converts each profile to a weighted feature vector and finds the closest vectors:

$$\text{similarity}(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$

Score range: 0–1 → scaled to 0–100%

**Why cosine over euclidean?**  
Cosine measures angle between vectors (lifestyle direction), not magnitude. 
Two students both spending different amounts but in the same relative proportions match better than absolute distance would suggest.

### Algorithm 2 — K-Means Clustering
Groups students into **lifestyle archetypes**:

```python
# Optimal K found by Silhouette Score (typically K=5-7 for 500 students)
for k in range(3, 11):
    km = KMeans(n_clusters=k, random_state=42)
    sil = silhouette_score(X, km.fit_predict(X))
```

Same-cluster students receive a **+3% compatibility bonus** in final scoring.

### Algorithm 3 — Weighted Feature Scoring
Interpretable per-feature breakdown:

```python
score = (0.30 * budget_score     # most important
       + 0.25 * sleep_score
       + 0.20 * study_score
       + 0.15 * cleanliness_score
       + 0.05 * year_score
       + 0.05 * schedule_score)
```

### Final Blend
```python
final = 0.60 * weighted_score + 0.40 * cosine_score * 100 + cluster_bonus
```

---

## 5. Training Process

Run: `python models/train_model.py`

Steps:
1. Load / generate dataset
2. MinMaxScaler fit on training profiles
3. KMeans trained with best K (silhouette)
4. Cosine similarity evaluated against 2000 labeled pairs
5. Metrics saved to `models/evaluation_report.json`
6. Artifacts: `scaler.pkl`, `kmeans_model.pkl`

---

## 6. Evaluation Metrics

| Metric | What it measures | Target |
|--------|-----------------|--------|
| MAE (Mean Absolute Error) | Avg error in compatibility score prediction | < 12 points |
| RMSE | Penalizes large errors more | < 18 points |
| Precision@5 | Are top-5 matches truly compatible? | > 0.75 (75%) |
| Silhouette Score | Quality of K-Means clusters | > 0.35 |

---

## 7. Why 100% Accuracy Is NOT Possible

Human compatibility is influenced by factors **outside structured data**:

1. **Personality traits** — introvert vs extrovert, communication style
2. **Cultural background** — food preferences, religious practices, festivals
3. **Dynamic habits** — stress during exams, relationship status changes
4. **Unmeasurable chemistry** — some people just click naturally
5. **Self-reporting bias** — students rate themselves as cleaner than they are

### Our Reliability Target
- **Short-term goal:** Precision@5 > 70% (7 out of 10 top recommendations are good)
- **Long-term goal:** Precision@5 > 85% via feedback retraining

### Iterative Improvement Loop
```
User gets matches → Rates them (👍/👎) → Feedback stored in Firestore
     ↓
Monthly retraining with real feedback labels
     ↓
Feature weight auto-optimization (Grid Search / Bayesian Opt)
     ↓
Better Precision@5 → better user satisfaction
```

---

## 8. Future Enhancements

| Enhancement | Impact |
|------------|--------|
| Collaborative Filtering | "Students like you liked X" |
| NLP on free-text preferences | Capture nuance in written responses |
| Matrix Factorization (SVD) | Better cold-start for new users |
| Neural Embeddings | Deep compatibility representations |
| A/B Testing framework | Validate model changes with real users |
